## До того как вы приступите к решению:
**Tools → Settings → Editor → completions / suggestions / linting → disable**

## Задание 1

Допишите 2 реализации функции `increment()`, которая **увеличивает глобальную переменную `counter` на 1**:

**с/без** (!) python синтаксического сахара. Сигнатуру функции менять нельзя.

**1.1: с python синтаксическим сахаром**

In [1]:
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')


counter=2 -- great!


**1.2: без python синтаксического сахара**

In [2]:
counter = 0

def increment():
    globals()['counter'] = globals()['counter'] + 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')


counter=2 -- great!


## Задание 2

Достаньте **только функцию `sqrt`** из модуля `math` и исполните sqrt(169).  
Нельзя исполнять `import math`.

Подготовьте 2 решения.

...

In [3]:
from math import sqrt as sqrt_from_math

result_1 = sqrt_from_math(169)

sqrt_from_import = __import__('math', fromlist=['sqrt']).sqrt
result_2 = sqrt_from_import(169)

result = result_2
assert result_1 == result_2 == 13
assert result == 13
print(result_1, result_2)


13.0 13.0


## Задание 3

Динамический импорт и перезагрузка.

1. Создайте модуль `mod.py`:

In [4]:
%%writefile mod.py
msg = "A"


Writing mod.py


2. Импортируйте его и выведите `msg`

3. Измените `msg` на `B` в файле

4. Без перезагрузки сессии ноутбука, выведите новое значение `msg`

In [5]:
import importlib
import sys
from pathlib import Path

sys.modules.pop('mod', None)
importlib.invalidate_caches()

mod = importlib.import_module('mod')
print(mod.msg)

Path('mod.py').write_text('msg = "B"\n# changed before reload\n', encoding='utf-8')
importlib.invalidate_caches()
mod = importlib.reload(mod)
print(mod.msg)


A
B


## Задание 4

У вас есть дирректория `pkg`:

In [6]:
!mkdir -p pkg

In [7]:
%%writefile pkg/m1.py
pi = 3.1415_92_65
_e = 2.7
__i = -1

Writing pkg/m1.py


```
pkg/
└── m1.py
```

Ниже ячейки для вашего кода, а после задание

### **Первый способ**: через модуль из стандартной библиотеки CPython

In [8]:
from pathlib import Path

Path('pkg/__init__.py').write_text(
    "from importlib import import_module\n"
    "_m1 = import_module('.m1', __name__)\n"
    "__all__ = [name for name in dir(_m1) if not name.startswith('_')]\n"
    "globals().update({name: getattr(_m1, name) for name in __all__})\n",
    encoding='utf-8',
)
print(Path('pkg/__init__.py').read_text(encoding='utf-8'))


from importlib import import_module
_m1 = import_module('.m1', __name__)
__all__ = [name for name in dir(_m1) if not name.startswith('_')]
globals().update({name: getattr(_m1, name) for name in __all__})



### **Второй способ**: в одну строчку без доп.модулей

In [9]:
open('pkg/__init__.py', 'w', encoding='utf-8').write("_m1 = __import__(__name__ + '.m1', fromlist=['*'])\n__all__ = [name for name in dir(_m1) if not name.startswith('_')]\nglobals().update({name: getattr(_m1, name) for name in __all__})\n")


182

### Текст задания:
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.  
Вам необходимо изменить структуру `pkg` пакета / содержимое его модулей, чтобы следующий код выполнялся корректно:

In [10]:
from pkg import *
pi

3.14159265

**Важно!**  
При обновлении любых данных в дирректории проекта, вам необходимо перезагружать сессию ipynb:  
`Runtime --> Restart Session` и перезапустить необходимые ячейки задания,  
иначе результаты могут быть для вас некорректными.

## Задание 5

При правильно решённом **задании 4** вам необходимо:
- изменить `pkg`
- дописать код ниже

так, чтобы "дотянуться" до `__i`.  
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.    
Если вы решите перезагрузить сессию, то для решения **задания 5** необходимо перезапустить ячейки **задания 4**.  

In [11]:
import importlib
import sys
from pathlib import Path

Path('pkg/__init__.py').write_text(
    "_m1 = __import__(__name__ + '.m1', fromlist=['*'])\n"
    "__all__ = [name for name in _m1.__dict__ "
    "if not (name.startswith('__') and name.endswith('__'))]\n"
    "globals().update({name: getattr(_m1, name) for name in __all__})\n",
    encoding='utf-8',
)

sys.modules.pop('pkg', None)
sys.modules.pop('pkg.m1', None)
importlib.invalidate_caches()


### Решение

In [12]:
from pkg import *
__i

-1

## Задание 6

Изменяемое замыкание. Почему этот код ведёт себя неожиданно? Исправьте.

In [13]:
def create_accumulators():
    # В исходном коде возвращался список чисел, а не список функций.
    # idx=i фиксирует текущее значение i для каждого замыкания.
    values = []
    accumulators = []
    for i in range(3):
        values.append(0)

        def accumulator(x, idx=i):
            values[idx] += x
            return values[idx]

        accumulators.append(accumulator)
    return accumulators

acc_list = create_accumulators()
print(acc_list[0](10))
print(acc_list[1](5))
print(acc_list[0](7))


10
5
17


## Задание 7
При перезапуске сессии ноутбука решение задачи начинается сначала

### 7.1: Востановите работу `print`, не используя `del`

In [14]:
print = 1

In [15]:
import builtins

print = builtins.print
print('print восстановлен без del')


print восстановлен без del


### 7.1: Удалите объект `print`, после востановите его функционал

In [16]:
print = 1
del print

import builtins

print = builtins.print
print('print восстановлен после del')


print восстановлен после del


## Задание 8

Замыкание с изменяемым состоянием. Создайте функцию-счётчик, которая запоминает количество вызовов между разными экземплярами:

In [17]:
def make_shared_counter():
    state = make_shared_counter.__dict__.setdefault('_state', {'count': 0})

    def counter():
        state['count'] += 1
        return state['count']

    return counter

c1 = make_shared_counter()
c2 = make_shared_counter()

print(c1())
print(c2())
print(c1())


1
2
3


## Задание 9

Допишите код, чтобы функция `outer` возвращала **словарь с тремя замыканиями**: `add()`, `mul()`, `get()` — работающими с одной и той же закрытой переменной `value`.

In [18]:
def outer(val=0):
    value = val

    def add(x):
        nonlocal value
        value += x
        return value

    def mul(x):
        nonlocal value
        value *= x
        return value

    def get():
        return value

    return {'add': add, 'mul': mul, 'get': get}

obj = outer(10)
obj['add'](5)
obj['mul'](2)
assert obj['get']() == 30
print(obj['get']())


30


## Задание 10

Создать closure, которая принимает функцию и возвращает новую функцию с кэшированием результатов (мемоизацией).

*Теоретическая справка:*

Функция `memoize` принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

В примере с функцией `fib` (числа Фибоначчи) мемоизация уменьшает количество рекурсивных вызовов с экспоненциального до линейного, так как результаты для каждого n вычисляются один раз.

Таким образом, мемоизация экономит время за счёт памяти — класическая оптимизация "время против памяти" — и особенно полезна для функций с дорогими вычислениями и повторяющимися входами.

Алгоритм для решения:

- Функция memoize принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

- Внутренняя функция wrapper проверяет, есть ли для данного входного аргумента (x) уже вычисленный результат в словаре cache.

- Если результат есть, то он возвращается из кеша, и вычисления не повторяются.

- Если нет, то вызывается исходная функция func(x), результат сохраняется в cache и возвращается.


In [19]:
def memoize(func):
    cache = {}

    def wrapper(x):
        if x not in cache:
            cache[x] = func(x)
        return cache[x]

    return wrapper

@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

print(fib(10))  # 55


55
